From an Google Search 'Word2Vec' (https://www.google.com/search?q=word2vec&sca_esv=e755c4fcff4cb9a6&rlz=1C1ONGR_enUS1065US1065&sxsrf=APpeQnse_RvzfuT5h5oe8j2BE1mcGgX8LA%3A1785529232767&ei=kANtatmzLrmXruEP-OKzgQk&biw=2874.6865234375&bih=1066.833740234375&sclient=gws-wiz-serp&fbs=ABfTbFVyMZGZf1hfvX9uKjN_-G8c4u0nXx4bEIpwm1lnNH832VstEKsVDqPorK0Gahnm2no1YAFtlsByIZaJlK7yr6gIShz8_nfnRyCFKBFanfbilXpMs-cznwqr4eRh15jLYnTY1jneHErIL1s8ylJ677g0-Yzv9SeiVzusgosrLmIdC_Li_URL_fqqHo09-SPTQ8fS7ou1p27xw7ju_YWEy8MXRBbSOQ&aep=10&ntc=1&mstk=AUtExfDjGCbGLkqwvITjmZJS3lyGsQh-otF9QlZqdVr5gU5RMD2IR2k1L8GgtstNNOkW8V58BHVtijYOvkBDi7NBM5A_oMG-Zf0baGofEpssl-8tXTTBHYJ-zAyQf1SsQaucVv5i4H8Y9dLE6rxIA0H8BWHQDetvUAovqUuPvmgMDxNYH-jXqmnl36Mo1P1SpuyQKJJ34T4c3k2XTIGzOnTpSBNSxkUATZYU0fZjdRK4CXHm30zQHWnNHqBKrc5SRmytvIXI2UnV4_CWRQ&aioh=3&csuir=1&cs=0&sourceid=chrome&ccb=1&hl=en-US&atvm=1&mtid=NgRtau_2NaPrmLQPuaTfsAg&udm=50)



In [1]:
# Step 1: Install the Necessary Libraries

!pip install transformers datasets scikit-learn pandas


In [3]:
# Step 2: Load a Pre-trained Medical BERT Model:

import torch
from transformers import AutoTokenizer, AutoModel

# Check that PyTorch sees your GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the medical BERT tokenizer and model (BioBERT)
model_name = "dmis-lab/biobert-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

print("✅ Medical BERT model successfully loaded onto your GPU!")


Using device: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Medical BERT model successfully loaded onto your GPU!


In [ ]:
# # Step 4: Create the Sample Dataset in Colab:

# import pandas as pd

# # Create a sample clinical dataset with a built-in anomaly
# data = {
#     'encounter_id': [101, 102, 103, 104, 105],
#     'specialty': ['Cardiology', 'Orthopedics', 'Pediatrics', 'Neurology', 'Gastroenterology'],
#     'clinical_notes': [
#         "Patient presents with intermittent chest pain and shortness of breath during exertion. EKG shows minor ST changes. Scheduled for a stress test.",
#         "A 45-year-old male with a closed fracture of the right radius following a fall. Closed reduction performed and fiberglass cast applied.",
#         "Routine 2-year well-child visit. Growth charts tracking at the 60th percentile. Immunizations updated. Normal developmental milestones met.",
#         "Patient complains of acute, severe crushing chest pain radiating to the left jaw, accompanied by diaphoresis and nausea. Emergency EKG ordered.", # ANOMALY: Cardiology text labeled as Neurology
#         "Subjective complaints of chronic abdominal bloating and epigastric burning after meals. Recommended upper endoscopy to rule out GERD or gastritis."
#     ]
# }

# df = pd.DataFrame(data)
# print(f"Dataset created successfully! Total records: {len(df)}")
# print(df[['encounter_id', 'specialty']])


In [5]:
# Step 4b: Create a Larger Sample Dataset in Colab:

import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Templates for realistic clinical notes across 5 specialties
templates = {
    'Cardiology': [
        "Patient complains of intermittent chest tightness and dyspnea on exertion. Scheduled for stress echo.",
        "Follow-up for benign hypertension. Blood pressure controlled on Lisinopril. Regular EKG shows normal sinus rhythm.",
        "A 62-year-old female presents with palpitations and fluttering sensations. Order 48-hour Holter monitor.",
        "History of CAD and stenting. Current symptoms stable. Continue daily Aspirin and Atorvastatin regimen."
    ],
    'Orthopedics': [
        "Patient presents with severe right knee pain following a twist during sports. Suspected meniscus tear, scheduled MRI.",
        "Evaluation of chronic lower back pain radiating down the left leg. Positive straight leg raise test.",
        "Follow-up for stable non-displaced fracture of the left fibula. X-ray shows good healing, transition to walking boot.",
        "A 34-year-old male with persistent shoulder impingement. Administered local corticosteroid injection to subacromial bursa."
    ],
    'Pediatrics': [
        "Routine 18-month well-child examination. Growth charts tracking well at 75th percentile. Vaccinations administered.",
        "Mother reports 3-day history of low-grade fever, runny nose, and tugging at ears. Exam confirms acute otitis media.",
        "Child presenting with acute dry barky cough and mild stridor. Diagnosed with mild croup, single dose of Decadron given.",
        "School-age child presenting for sports physical. Normal cardiovascular, respiratory, and musculoskeletal exam."
    ],
    'Neurology': [
        "Patient describes worsening chronic migraines occurring 3-4 times per week. Initiating Topamax for prophylaxis.",
        "Evaluation of progressive resting tremor in right hand accompanied by bradykinesia. Consistent with early Parkinsonism.",
        "A 55-year-old female presents with brief episodes of facial numbness and tingling. Schedule brain MRI with contrast.",
        "Follow-up for well-controlled epilepsy. Patient reports being seizure-free for 12 months on Levetiracetam."
    ],
    'Gastroenterology': [
        "Patient reports persistent epigastric burning and acid regurgitation worse at night. Prescribed Omeprazole.",
        "Follow-up for moderate Crohn's disease. Patient reports clinical remission on current biologic therapy.",
        "A 48-year-old male presenting with alternating constipation and diarrhea, accompanied by abdominal cramping.",
        "Scheduled for screening colonoscopy due to strong family history of colon cancer. Preparation instructions provided."
    ]
}

# Generate 97 normal records
specialties_pool = list(templates.keys())
generated_data = []

# Generate sequential IDs starting at 101
for i in range(97):
    enc_id = 101 + i
    # Ensure we don't accidentally claim our reserved anomaly IDs
    if enc_id in [104, 135, 172]:
        enc_id += 300

    spec = np.random.choice(specialties_pool)
    note = np.random.choice(templates[spec])

    # Add minor random noise to text variations so they aren't identical copies
    noise = f" Vital signs stable. Patient to follow up in {np.random.choice([2, 4, 6])} weeks."
    generated_data.append({'encounter_id': enc_id, 'specialty': spec, 'clinical_notes': note + noise})

# Form the initial DataFrame
df = pd.DataFrame(generated_data)

# Inject our 3 explicit anomalies with the exact target encounter IDs
anomalies = [
    {
        'encounter_id': 104,
        'specialty': 'Neurology',
        'clinical_notes': "Patient complains of acute, severe crushing chest pain radiating to the left jaw, accompanied by diaphoresis and nausea. Emergency EKG ordered."
    }, # ERROR: Clear Cardiology narrative mislabeled as Neurology
    {
        'encounter_id': 135,
        'specialty': 'Pediatrics',
        'clinical_notes': "A 78-year-old male presents with severe progressive resting tremor in the right hand and shuffling gait. Family notes memory decline."
    }, # ERROR: Clear Geriatric Neurology narrative mislabeled as Pediatrics
    {
        'encounter_id': 172,
        'specialty': 'Orthopedics',
        'clinical_notes': "Subjective complaints of chronic abdominal bloating, sharp epigastric burning after meals, and dark tarry stools. Scheduled urgent upper endoscopy."
    }  # ERROR: Clear Gastroenterology narrative mislabeled as Orthopedics
]

# Append anomalies and reset the dataframe index
df = pd.concat([df, pd.DataFrame(anomalies)], ignore_index=True)
df = df.sort_values(by='encounter_id').reset_index(drop=True)

print(f"📊 Dataset created successfully! Total records: {len(df)}")
print("\nTarget Anomalies Hidden Inside the Dataset:")
print(df[df['encounter_id'].isin([104, 135, 172])][['encounter_id', 'specialty']])

📊 Dataset created successfully! Total records: 100

Target Anomalies Hidden Inside the Dataset:
    encounter_id    specialty
3            104    Neurology
34           135   Pediatrics
71           172  Orthopedics


In [6]:
import torch
import numpy as np

# Ensure your model from the previous step is active
# model.eval() # This line should ideally be called once, outside the function or in an earlier setup cell

def get_bert_embedding(text):
    # Tokenize text and move tensors to the GPU
    inputs = tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Extract the 'Pooler Output' which represents the semantic meaning of the entire text block
    embeddings = outputs.pooler_output.cpu().numpy()
    return embeddings[0]

# Step 4.5  Create a unified text field for the model to read (inserted in later)

df['combined_text'] = "Specialty: " + df['specialty'] + " | Notes: " + df['clinical_notes']

# Function to get embeddings (using the function we defined in the previous step)
print("Generating contextual embeddings for combined text...")
embeddings_list = []
for text in df['combined_text']:
    # Get vector and squeeze to flatten from (1, 768) to (768,)
    vector = get_bert_embedding(text).squeeze()
    embeddings_list.append(vector)

# Convert list to a clean 2D NumPy array for machine learning
X = np.array(embeddings_list)
print(f"Matrix shape: {X.shape} (5 records, 768 mathematical features each)")

Generating contextual embeddings for combined text...
Matrix shape: (100, 768) (5 records, 768 mathematical features each)


In [ ]:

print(X)
type(X)

In [7]:
# Step 5: Convert Text into Medical BERT Embeddings:

import torch
import numpy as np

# Ensure your model from the previous step is active
model.eval()

def get_bert_embedding(text):
    # Tokenize text and move tensors to the GPU
    inputs = tokenizer(text, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Extract the 'Pooler Output' which represents the semantic meaning of the entire text block
    embeddings = outputs.pooler_output.cpu().numpy()
    return embeddings[0]

# Generate vectors for all clinical notes
print("Processing text through BioBERT on the GPU...")
df['vector'] = df['clinical_notes'].apply(get_bert_embedding)
print("✅ Text converted into dense math vectors successfully!")



Processing text through BioBERT on the GPU...
✅ Text converted into dense math vectors successfully!


In [8]:
# Step 6: Detect the Anomaly Using Isolation Forest:


from sklearn.ensemble import IsolationForest

# Initialize Isolation Forest
# contamination=0.20 tells the AI we expect roughly 20% of our data (1 out of 5) to be anomalous
# iso_forest = IsolationForest(contamination=0.20, random_state=42)

# Initialize the forest WITHOUT a hardcoded contamination percentage
iso_forest_raw = IsolationForest(contamination='auto', random_state=42)
iso_forest_raw.fit(X)

# Train the model on our BERT vectors and predict outliers
# Predictions return: 1 for normal data, -1 for anomalies
df['anomaly_score'] = iso_forest_raw.predict(X)

# Display the results
print("\n--- DETECTED ANOMALIES ---")
for index, row in df.iterrows():
    status = "⚠️ ANOMALY DETECTED" if row['anomaly_score'] == -1 else "✅ Normal Record"
    print(f"\nEncounter ID: {row['encounter_id']} | Status: {status}")
    print(f"Assigned Specialty: {row['specialty']}")
    print(f"Notes snippet: {row['clinical_notes'][:80]}...")

# Extract raw anomaly scores (lower/more negative = more anomalous)
# Note: sklearn offsets the score so that negative numbers are outliers
df['raw_anomaly_score'] = iso_forest_raw.score_samples(X)

# Sort records to show the most suspicious ones at the very top
df_sorted = df.sort_values(by='raw_anomaly_score')
print("\n--- SORTED RECORDS ---")
print(df_sorted[['encounter_id', 'specialty', 'raw_anomaly_score']])


--- DETECTED ANOMALIES ---

Encounter ID: 101 | Status: ✅ Normal Record
Assigned Specialty: Neurology
Notes snippet: Patient describes worsening chronic migraines occurring 3-4 times per week. Init...

Encounter ID: 102 | Status: ✅ Normal Record
Assigned Specialty: Pediatrics
Notes snippet: School-age child presenting for sports physical. Normal cardiovascular, respirat...

Encounter ID: 103 | Status: ⚠️ ANOMALY DETECTED
Assigned Specialty: Gastroenterology
Notes snippet: A 48-year-old male presenting with alternating constipation and diarrhea, accomp...

Encounter ID: 104 | Status: ⚠️ ANOMALY DETECTED
Assigned Specialty: Neurology
Notes snippet: Patient complains of acute, severe crushing chest pain radiating to the left jaw...

Encounter ID: 105 | Status: ✅ Normal Record
Assigned Specialty: Pediatrics
Notes snippet: School-age child presenting for sports physical. Normal cardiovascular, respirat...

Encounter ID: 106 | Status: ✅ Normal Record
Assigned Specialty: Neurology
Notes snip

**Dynamically Setting the Threshold. **Once you have thousands of records, you cannot look at them one by one. You use statistical rules of thumb to isolate the threshold programmatically:


*   The Standard Deviation Rule (Extreme Outliers): Calculate the average score of your dataset. Flag any record that sits more than 2 or 3 standard deviations away from that average.
*   The Percentile Method for Auditing Budgets: If your quality assurance department only has the manpower to review 50 records a week, you simply sort your database by the lowest raw_anomaly_score and pull the Top 50 worst scores, regardless of percentage.








In [ ]:
# # Step 7: Try This Statistical Thresholding Script

# import numpy as np

# scores = df['raw_anomaly_score'].values
# mean_score = np.mean(scores)
# std_score = np.std(scores)

# # Define a statistical threshold (e.g., 1 standard deviation away for this tiny dataset)
# # For large datasets, 2 or 3 standard deviations is standard
# threshold = mean_score - (1.0 * std_score)

# df['dynamic_anomaly_flag'] = df['raw_anomaly_score'].apply(lambda x: "⚠️ ANOMALY" if x < threshold else "✅ Normal")

# print(f"Calculated Dataset Mean Score: {mean_score:.4f}")
# print(f"Statistical Anomaly Cutoff Threshold: {threshold:.4f}\n")
# print(df[['encounter_id', 'specialty', 'raw_anomaly_score', 'dynamic_anomaly_flag']])


In [11]:
# Step 7: Try This Statistical Thresholding Script
import numpy as np
import textwrap
from google.colab import data_table

# 1. Calculate statistical metrics
scores = df['raw_anomaly_score'].values
mean_score = np.mean(scores)
std_score = np.std(scores)

# Define a statistical threshold (Using 2 standard deviations)
threshold = mean_score - (2.0 * std_score)

# Apply flags to the dataframe
df['dynamic_anomaly_flag'] = df['raw_anomaly_score'].apply(lambda x: "⚠️ ANOMALY" if x < threshold else "✅ Normal")

# 2. Print high-level summary metrics
print("==================================================")
print(f"📊 SYSTEM METRICS SUMMARY")
print("==================================================")
print(f"Calculated Dataset Mean Score: {mean_score:.4f}")
print(f"Statistical Anomaly Cutoff Threshold: {threshold:.4f}\n")

# 3. Explicitly list all flagged cases first with 80-character line wrapping
print("==================================================")
print("⚠️ FLAGGED ANOMALY ENCOUNTERS (FOR HUMAN REVIEW)")
print("==================================================")
anomalies_only = df[df['dynamic_anomaly_flag'] == "⚠️ ANOMALY"]

if len(anomalies_only) > 0:
    for index, row in anomalies_only.iterrows():
        print(f"• ID: {row['encounter_id']} | Specialty: {row['specialty']} | Score: {row['raw_anomaly_score']:.4f}")

        # Wrap text to 80 characters and indent subsequent lines for cleaner reading
        wrapped_notes = textwrap.fill(row['clinical_notes'], width=80, initial_indent="  Notes: ", subsequent_indent="         ")
        print(f"{wrapped_notes}\n")
else:
    print("No anomalies detected under the current statistical threshold.\n")

# 4. Enable and display the entire dataset in a scrollable format
print("==================================================")
print("📋 COMPLETE ENCOUNTER DATASET (SCROLLABLE TABLE)")
print("==================================================")

# Configure Colab's interactive table output settings
data_table.enable_dataframe_formatter()
display(df[['encounter_id', 'specialty', 'raw_anomaly_score', 'dynamic_anomaly_flag']])


📊 SYSTEM METRICS SUMMARY
Calculated Dataset Mean Score: -0.4758
Statistical Anomaly Cutoff Threshold: -0.5741

⚠️ FLAGGED ANOMALY ENCOUNTERS (FOR HUMAN REVIEW)
• ID: 104 | Specialty: Neurology | Score: -0.7479
  Notes: Patient complains of acute, severe crushing chest pain radiating to the
         left jaw, accompanied by diaphoresis and nausea. Emergency EKG ordered.

• ID: 135 | Specialty: Pediatrics | Score: -0.7581
  Notes: A 78-year-old male presents with severe progressive resting tremor in
         the right hand and shuffling gait. Family notes memory decline.

• ID: 172 | Specialty: Orthopedics | Score: -0.5960
  Notes: Subjective complaints of chronic abdominal bloating, sharp epigastric
         burning after meals, and dark tarry stools. Scheduled urgent upper
         endoscopy.

📋 COMPLETE ENCOUNTER DATASET (SCROLLABLE TABLE)


,encounter_id,specialty,raw_anomaly_score,dynamic_anomaly_flag
0,101,Neurology,-0.421847,✅ Normal
1,102,Pediatrics,-0.473615,✅ Normal
2,103,Gastroenterology,-0.507246,✅ Normal
3,104,Neurology,-0.747889,⚠️ ANOMALY
4,105,Pediatrics,-0.473615,✅ Normal
...,...,...,...,...
95,196,Cardiology,-0.464550,✅ Normal
96,197,Pediatrics,-0.476929,✅ Normal
97,404,Pediatrics,-0.463766,✅ Normal
98,435,Neurology,-0.416418,✅ Normal


In [ ]:
# Step 8: This is a correction script, to suggest to which specialty a record should be assigned to
# in this case, for illustration, it uses the known value of the anomalous case (4), but could
# be modified to pull case record numbers whose values were based on thresholds.

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Step 1: Isolate our known "good" clean records to build baseline profiles
# We exclude Encounter 4 because we already know it is an anomaly
clean_df = df[df['encounter_id'] != 104] # Changed from 4 to 104

# Create a dictionary to store the baseline vector for each specialty
specialty_baselines = {}
for spec in clean_df['specialty'].unique():
    # Get all vectors belonging to this specialty
    spec_vectors = np.array(clean_df[clean_df['specialty'] == spec]['vector'].tolist())
    # Calculate the average vector profile for this specialty
    specialty_baselines[spec] = np.mean(spec_vectors, axis=0).reshape(1, -1)

# Step 2: Grab the vector of our anomalous record (Encounter 104)
anomaly_vector = df[df['encounter_id'] == 104]['vector'].values[0].reshape(1, -1) # Changed from 4 to 104

# Step 3: Calculate similarity between the anomaly and every baseline profile
print("--- COGSINE SIMILARITY ANALYSIS FOR ENCOUNTER 104 ---") # Changed from 4 to 104
print(f"Current Mislabeled Specialty: {df[df['encounter_id'] == 104]['specialty'].values[0]}\n") # Changed from 4 to 104

highest_score = -1
recommended_specialty = None

for spec, baseline_vector in specialty_baselines.items():
    # Calculate cosine similarity (returns a 2D array, so we grab the scalar value)
    similarity_score = cosine_similarity(anomaly_vector, baseline_vector)[0][0]
    print(f"Similarity to {spec} baseline: {similarity_score:.4f}")

    if similarity_score > highest_score:
        highest_score = similarity_score
        recommended_specialty = spec

print("\n--- SYSTEM RECOMMENDATION ---")
print(f"💡 This record looks like an error. It should be re-assigned to: **{recommended_specialty}**")
print(f"Confidence score: {highest_score:.4f}")

Open this window, and type "Resume medical anomaly project,"

https://www.google.com/search?q=word2vec&sca_esv=e755c4fcff4cb9a6&rlz=1C1ONGR_enUS1065US1065&sxsrf=APpeQnse_RvzfuT5h5oe8j2BE1mcGgX8LA%3A1785529232767&ei=kANtatmzLrmXruEP-OKzgQk&biw=2874.6865234375&bih=1066.833740234375&sclient=gws-wiz-serp&fbs=ABfTbFVyMZGZf1hfvX9uKjN_-G8c4u0nXx4bEIpwm1lnNH832VstEKsVDqPorK0Gahnm2no1YAFtlsByIZaJlK7yr6gIShz8_nfnRyCFKBFanfbilXpMs-cznwqr4eRh15jLYnTY1jneHErIL1s8ylJ677g0-Yzv9SeiVzusgosrLmIdC_Li_URL_fqqHo09-SPTQ8fS7ou1p27xw7ju_YWEy8MXRBbSOQ&aep=10&ntc=1&mstk=AUtExfCORGHfCOoAuq7N4r7jnULryoQP9ma9pC4jQQXipsGY5RxDx0SoU3-_i6yEwlmdL02lpsDg5WHN79RU6d1S282tbqz_6-uzWKm5h9aoEYdfP06eb30U-uV41FDy0NOlRS5qIqTeaZ56PXp6UsOfDSXoXjG0GN1TmSqmO6nuaVsluOGtn9inSl004f0U0f-fpjbAjYYzyTf8daJsfBTjIgUu5T5CImeclcIlje7YHXWOl9rOozVfpK7t-elnPHzcIoA7Bkz3MEvI2A&aioh=3&csuir=1&cs=0&sourceid=chrome&ccb=1&hl=en-US&atvm=1&mtid=NgRtau_2NaPrmLQPuaTfsAg&udm=50